# Trabajo de migración de la lista del SII

Puntos:
1. Se impoportan la base
2. Se importan los datos
3. Se dejan solo empresas que estén funcionando
4. Se dejan empresas con más de {} trabajadores
5. Se deja en un CSV

In [ ]:
# Instalar dependencias si no están disponibles
import importlib, subprocess, sys
import pandas as pd

def ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"{pkg} instalado.")
    else:
        print(f"{pkg} ya disponible.")

ensure("anthropic")
ensure("pandas")

anthropic ya disponible.
pandas ya disponible.


In [34]:
# Empresas personas jurídicas 2024
df_empresas = pd.read_csv(
    "/Users/juanpablozepeda/Proyecto Plutto /Sii folder/PUB_EMPRESAS_PJ_2024.txt",
    sep="\t",
    encoding="utf-8",
    dtype={"RUT": str, "DV": str},
)
# print(f"Empresas PJ — Filas: {len(df_empresas):,}")
# print(f"Columnas: {list(df_empresas.columns)}")

In [35]:
# Filtrar empresas sin fecha de término de giro

df_funcionando = df_empresas[df_empresas["Fecha término de giro"].isna()].copy()

print(f"Sin termino de giro:    {len(df_funcionando):,}")
print(f"con término de giro:    {df_empresas['Fecha término de giro'].notna().sum():,}")
print(f"Total:                  {len(df_empresas):,}")

#df_funcionando[["RUT","DV", "Razón social",'Número de trabajadores dependie',  "Tramo según ventas", "Rubro económico", "Actividad económica", "Región", "Provincia"]].head(20)

Sin termino de giro:    978,601
con término de giro:    15,875
Total:                  994,476


In [36]:
# Construcción del DF final con columnas seleccionadas

# Unir RUT + DV en formato "RUT-DV"
df_funcionando["RUT_DV"] = df_funcionando["RUT"].str.strip() + "-" + df_funcionando["DV"].str.strip()

# Detectar nombre exacto de "Número de trabajadores" (puede estar truncado)
col_trabajadores = next((c for c in df_funcionando.columns if "trabajadores" in c.lower()), None)

# Detectar nombre exacto de "Tipo de contribuyente"
col_tipo = next((c for c in df_funcionando.columns if "tipo" in c.lower() and "contribuy" in c.lower()), None)

print(f"Col trabajadores → {col_trabajadores}")
print(f"Col tipo de contribuyente → {col_tipo}")

# Columnas a mantener (en orden)
cols_finales = ["RUT_DV", "Razón social", "Tramo según ventas", col_trabajadores, col_tipo, "Actividad económica", "Región"]
cols_finales = [c for c in cols_finales if c is not None]  # eliminar None si no existe la columna

df_final = df_funcionando[cols_finales].copy()

# Renombrar para mayor claridad
rename_map = {
    col_trabajadores: "Número de trabajadores",
}
if col_tipo:
    rename_map[col_tipo] = "Tipo de contribuyente"

df_final = df_final.rename(columns=rename_map)

print(f"\nFilas: {len(df_final):,}")
print(f"Columnas: {list(df_final.columns)}")
df_final.head(10)

Col trabajadores → Número de trabajadores dependie
Col tipo de contribuyente → Tipo de contribuyente

Filas: 978,601
Columnas: ['RUT_DV', 'Razón social', 'Tramo según ventas', 'Número de trabajadores', 'Tipo de contribuyente', 'Actividad económica', 'Región']


,RUT_DV,Razón social,Tramo según ventas,Número de trabajadores,Tipo de contribuyente,Actividad económica,Región
0,50000100-3,FUNDACION ARTURO IRARRAZAVAL CORREA,1,5,SIN PER. JURIDICA,OTRAS ACTIVIDADES DE ATENCION EN INSTITUCIONES,XIII REGION METROPOLITANA
1,50000510-6,TORREJON CASTRO LUIS Y JOSE,1,0,SIN PER. JURIDICA,TRANSPORTE DE CARGA POR CARRETERA,IV REGION COQUIMBO
2,50000710-9,PONCE PONCE MANUEL E HIJOS,2,0,SIN PER. JURIDICA,FABRICACION DE PRODUCTOS METALICOS PARA USO ES...,IV REGION COQUIMBO
3,50001570-5,GARCIA MARTINEZ MARIA MILAGRO Y OTRO,1,0,SIN PER. JURIDICA,ACTIVIDADES DE RESTAURANTES Y DE SERVICIO MOVI...,XIII REGION METROPOLITANA
4,50001700-7,COMUNIDAD EDIFICIO MOSQUETO 520,1,1,SIN PER. JURIDICA,CONSEJO DE ADMINISTRACION DE EDIFICIOS Y CONDO...,XIII REGION METROPOLITANA
5,50001900-K,COMUNIDAD EDIFICIO HUERFANOS 518,1,1,SIN PER. JURIDICA,CONSEJO DE ADMINISTRACION DE EDIFICIOS Y CONDO...,XIII REGION METROPOLITANA
6,50002100-4,COMUNIDAD EDIFICIO HUERFANOS 1022,1,9,SIN PER. JURIDICA,CONSEJO DE ADMINISTRACION DE EDIFICIOS Y CONDO...,XIII REGION METROPOLITANA
7,50002420-8,MARTINEZ ZARZUELA DANIEL Y OTRO,6,9,SIN PER. JURIDICA,ACTIVIDADES DE RESTAURANTES Y DE SERVICIO MOVI...,XIII REGION METROPOLITANA
9,50003040-2,DIAZ BRUNA ARMANDO Y OTRO,1,0,SIN PER. JURIDICA,VENTA AL POR MENOR DE BEBIDAS ALCOHOLICAS Y NO...,XIII REGION METROPOLITANA
10,50003920-5,SALHE HARCHA NICOLAS Y HNOS,1,0,SIN PER. JURIDICA,VENTA AL POR MENOR DE PRENDAS Y ACCESORIOS DE ...,XIII REGION METROPOLITANA


In [37]:
# Empresas con tramo según ventas > 8 (medians, grandes y muy grandes)

df_grandes = df_final[df_final["Tramo según ventas"] > 8].copy()

print(f"Empresas con tramo > 8: {len(df_grandes):,}")
# print()
print(df_grandes["Tramo según ventas"].value_counts().sort_index())


Empresas con tramo > 8: 30,746
Tramo según ventas
9     13157
10     7536
11     6074
12     1371
13     2608
Name: count, dtype: int64


In [38]:
#Emrpesas que tienen más de Min_Workers fundionando

min_workers = 10

df_funcionando_workers = df_grandes[df_grandes["Número de trabajadores"] > min_workers].copy()

print(f"Empresas que tinen más de {min_workers} trabajadores: {len(df_funcionando_workers):,}")

Empresas que tinen más de 10 trabajadores: 21,143


In [53]:
# Segmentación:

# Palabras claves 

SEGMENTOS = {
    "Mining / Energía": [
        "mina", "miner", "cantera", "extrac", "petroleo", "carbon de piedra",
        "cobre", "litio", "hierro", "zinc", "plomo", "mangane", "uranio",
        "plata", "oro", "gas natural", "refinacion del petroleo",
        "generacion de energia", "transmision de energia", "distribucion de energia",
        "generacion, transmision", "gasoducto", "oleoducto", "regasificacion",
        "extraccion de sal", "turba", "carbon vegetal",
        "apoyo para la extraccion", "apoyo para la explotacion de otras minas",
        "explotacion de minas",
    ],
    "Financiero": [
        "bancar", "bancaria", "intermediacion monetaria", "leasing financiero",
        "factoring", "financiera", "financiero", "corredores de bolsa",
        "agentes de valores", "fondo de inversion", "fondos de inversion",
        "fondo mutuo", "administradoras de fondos", "sociedad de cartera",
        "sociedades de cartera", "seguro de vida", "seguros generales", "reaseguro",
        "isapre", "afp", "casas de cambio", "camara de compensacion",
        "credito prendario", "clasificadora de riesgo", "securitizadora",
        "cajas de compensacion", "agencias de calificacion crediticia",
        "agencias de cobro", "administracion de tarjetas", "administracion de mercados financieros",
        "otras actividades de servicios financieros",
        "otras actividades auxiliares de las actividades de servicios financieros",
        "otras actividades auxiliares de las actividades de seguros",
        "otras actividades de concesion de credito",
        "fondos y sociedades de inversion",
        "asesoria y gestion en la compra o venta de pequeñas y medianas empresas",
        "empresas de asesoria y consultoria en inversion financiera",
        "evaluacion de riesgos y daños",
    ],
    "Utilities / Infraestructura": [
        "suministro de agua", "captacion, tratamiento y distribucion de agua",
        "evacuacion de aguas", "aguas servidas", "aguas residuales",
        "suministro de electricidad", "distribucion de energia electrica",
        "transmision de energia electrica", "generacion de energia electrica",
        "fabricacion de gas", "distribucion de combustibles gaseosos",
        "suministro de vapor",
        "transporte de carga", "transporte de pasajeros", "transporte maritimo",
        "transporte interurbano", "transporte urbano", "transporte suburbano",
        "transporte por ferrocarril", "transporte por gasoducto", "transporte por oleoducto",
        "transporte por tuberias", "transporte por vias de navegacion",
        "transporte de carga maritimo", "servicios prestados por concesionarios de carreteras",
        "manipulacion de la carga", "almacenamiento y deposito",
        "explotacion de terminales terrestres",
        "telefonia", "telecomunicaciones", "television de pago", "transmisiones de radio",
        "programacion y transmisiones de television", "radiocomunicaciones",
        "portales web", "procesamiento de datos",
        "servicios vinculadas al transporte aereo", "servicios vinculadas al transporte acuatico",
        "agencias de naves",
        "concesionaria", "aeropuerto",             
        "servicios profesionales de ingenieria",    
        "ingenieria y construccion",
    ],
    "Manufactura / Retail": [
        "fabricacion de", "industrias basicas", "fundicion", "forja",
        "elaboracion de", "elaboracion y conservacion",
        "construccion de edificios", "construccion de carreteras",
        "construccion de otras obras", "construccion de proyectos",
        "construccion de buques", "terminacion y acabado de edificios",
        "demolicion", "preparacion del terreno", "instalaciones electricas",
        "instalaciones de gasfiteria", "otras instalaciones para obras",
        "otras actividades especializadas de construccion",
        "aserrado y acepilladura", "curtido y adobo",
        "venta al por menor en comercios de alimentos",
        "venta al por menor en comercios de vestuario",
        "venta al por mayor de alimentos", "venta al por mayor de bebidas",
        "venta al por mayor de carne",
        "mantenimiento y reparacion de vehiculos automotores",
        "venta de vehiculos automotores", "venta al por mayor de vehiculos",
        "fabricacion de vehiculos", "fabricacion de partes, piezas y accesorios para vehiculos",
        "fabricacion de carrocerias",
        "otras industrias manufactureras",
        "impresion", "reproduccion de grabaciones",
        "tratamiento y revestimiento de metales",
        "pesca", "acuicultura",
        "cultivo y crianza de peces", "cultivo de peces", "crianza de peces",
        "productos del mar", "conservacion de pescado",
        "elaboracion y conservacion de pescado",
        "elaboracion y conservacion de crustaceos",
        "procesamiento de pescado", "harina de pescado", "aceite de pescado",
    ],
}

def clasificar(actividad: str) -> str | None:
    a = actividad.lower()
    for segmento, keywords in SEGMENTOS.items():
        for kw in keywords:
            if kw in a:
                return segmento
    return None



# ── 3. Aplicar clasificación ──────────────────────────────────────────────────
df_funcionando_workers["Segmento"] = df_funcionando_workers["Actividad económica"].apply(clasificar)

#df_funcionando.head()

In [54]:
#Dejo solo los segmentados

df_segmentado = df_funcionando_workers[df_funcionando_workers["Segmento"].notna()].copy()

print(f"Empresas que tienen un segmento buscado: {len(df_segmentado):,}")

print(df_segmentado["Segmento"].value_counts().sort_index())

pd.crosstab(
    df_segmentado["Tramo según ventas"],
    df_segmentado["Segmento"],
    margins=True,
    margins_name="Total"
)
df_segmentado



Empresas que tienen un segmento buscado: 8,049
Segmento
Financiero                      642
Manufactura / Retail           4700
Mining / Energía                966
Utilities / Infraestructura    1741
Name: count, dtype: int64


,RUT_DV,Razón social,Tramo según ventas,Número de trabajadores,Tipo de contribuyente,Actividad económica,Región,Segmento
72,50075620-9,COMERCIAL EBERLEIN SPA,9,48,SIN PER. JURIDICA,MANTENIMIENTO Y REPARACION DE VEHICULOS AUTOMO...,V REGION VALPARAISO,Manufactura / Retail
145,50159080-0,INDUSTRIAS CELTA SPA,13,348,SIN PER. JURIDICA,"FABRICACION DE COLCHONES, FABRICACION DE OTROS...",IV REGION COQUIMBO,Manufactura / Retail
262,50314960-5,H M INSTALACIONES ELECTRICAS SPA,10,200,PERSONA JURIDICA COMERCIAL,INSTALACIONES ELECTRICAS,XIII REGION METROPOLITANA,Manufactura / Retail
355,50413920-4,CASTHER SPA.,9,79,PERSONA JURIDICA COMERCIAL,OTRAS ACTIVIDADES DE TRANSPORTE DE PASAJEROS P...,III REGION DE ATACAMA,Utilities / Infraestructura
1321,52000288-K,FORESTAL Y ASERRADEROS LEONERA LIMITADA.,12,225,PERSONA JURIDICA COMERCIAL,ASERRADO Y ACEPILLADURA DE MADERA,XVI REGION DE ÑUBLE,Manufactura / Retail
...,...,...,...,...,...,...,...,...
994398,99597200-K,FRIGORIFICO LAS BRASAS S.A.,11,94,PERSONA JURIDICA COMERCIAL,VENTA AL POR MAYOR DE CARNE Y PRODUCTOS CARNICOS,I REGION DE TARAPACA,Manufactura / Retail
994404,99597320-0,ADMINISTRADOR FINANCIERO DE TRANSANTIAGO S A,10,22,PERSONA JURIDICA COMERCIAL,OTROS TIPOS DE INTERMEDIACION MONETARIA N.C.P.,XIII REGION METROPOLITANA,Financiero
994416,99597800-8,NAVIERA AUSTRAL S.A.,11,329,PERSONA JURIDICA COMERCIAL,TRANSPORTE DE PASAJEROS MARITIMO Y DE CABOTAJE,X REGION LOS LAGOS,Utilities / Infraestructura
994424,99598020-7,"MUELLAJE DEL LOA S.A,",10,296,PERSONA JURIDICA COMERCIAL,ACTIVIDADES DE SERVICIOS VINCULADAS AL TRANSPO...,II REGION DE ANTOFAGASTA,Utilities / Infraestructura


## Acá parto probando el Scoring

# No encontradas en SII:

```  Jetti Resources LLC"         → empresa extranjera (LLC)
 "Compañía Minera Volcán S.A.A."→ empresa peruana (S.A.A.)
 "EDF Renewables Chile SpA"    → no encontrada (existe EDF ANDES SPA 76360893-K)
 "Conexión Energía S.A."       → no encontrada
 "Skretting Chile"             → no encontrada
 "Marley Coffee Chile SpA"     → no encontrada
 "Klap SpA"                    → no encontrada
 "Grand Leasing S.A."          → no encontrada
 "Kapin Capital SpA"           → no encontrada
 "PayCompany SpA"              → no encontrada
 "Sonacol"                     → no encontrada
 "OnNet Fibra Chile S.A."      → no encontrada
 "Clínica ACHS Salud S.A."     → no encontrada como entidad separada
 "Isapre Esencial S.A."        → no encontrada
 "CIAL Alimentos S.A."         → no encontrada
 "Gasco S.A."                  → no encontrada como entidad principal
 "Alta S.A."                   → no encontrada
 "Megacentro S.A."             → no encontrada
 ```


In [55]:

empresas_target = {
    "78127000-8": "Teck Resources Chile Ltda.",
    "76602739-3": "Minera Salar Blanco S.A.",
    "90266000-3": "Enaex S.A.",
    "76257379-2": "Interchile S.A.",
    "93458000-1": "Arauco S.A.",
    "78803130-0": "Magotteaux Andino S.A.",
    "76167834-5": "SK Godelius Ltda.",
    "99598300-1": "Sigdo Koppers S.A.",
    "96786880-9": "Sacyr Chile S.A.",
    "81458500-K": "Cámara Chilena de la Construcción A.G.",
    "77078244-9": "Xepelin SpA",               # SII: X CAPITAL SPA
    "97006000-6": "Banco de Crédito e Inversiones",
    "97018000-1": "Scotiabank Chile S.A.",
    "96521680-4": "Redbanc S.A.",
    "76415528-9": "Buda.com SpA",
    "77126383-6": "Toku SpA",
    "76596744-9": "Chita SpA",
    "76229636-5": "Financia Capital SpA",
    "99595990-9": "Latam Trade Capital SpA",
    "76833300-9": "Essbio S.A.",
    "76000739-0": "Esval S.A.",
    "91806000-6": "Abastible S.A.",
    "96721360-8": "GasAndes S.A.",              # SII: GASODUCTO GASANDES S A
    "96636520-K": "Gasmar S.A.",
    "61219000-3": "Empresa de Transporte de Pasajeros Metro S.A.",
    "82777100-7": "DP World Chile S.A.",
    "96785680-0": "Puerto Coronel S.A.",
    "96602640-5": "Puerto Ventanas S.A.",
    "76466068-4": "Nuevo Pudahuel S.A.",
    "96684580-5": "Ferrocarril del Pacífico S.A.",
    "92580000-7": "Empresa Nacional de Telecomunicaciones S.A.",
    "99579260-5": "Asociación Chilena de Seguridad",
    "96802690-9": "Masisa S.A.",
    "90749000-9": "Falabella S.A.",
    "76101812-4": "Soprole S.A.",
    "90331000-6": "Cristalerías de Chile S.A.",
    "86740500-3": "Toyota Chile S.A.",
    "96856360-2": "Inchcape Chile S.A.",        # SII: INCHCAPE AUTOMOTRIZ CHILE S.A.
    "96556930-8": "Pluxee Chile S.A.",
    "80860400-0": "Blumar S.A.",
    "96926970-8": "Cooke Aquaculture Chile S.A.",
    "83628100-4": "Sonda S.A.",
    "76437608-0": "Aliservice S.A.",
    "76364071-K": "NS Agro S.A.",
    "76320186-4": "Tecno Fast S.A.",
    "86547900-K": "Viña Santa Rita S.A.",       # SII: SOC ANONIMA VINA SANTA RITA
    "76054451-5": "Arrimaq SpA",                # SII: ARRIMAQ LIMITADA
    "81698900-0": "Pontificia Universidad Católica de Chile",
    "71540800-7": "Universidad de Las Américas S.A.",
    "71500500-K": "Universidad Mayor",
    "70017820-K": "Cámara de Comercio de Santiago A.G.",
    "99527440-K": "Ksec SpA",
    "76727442-4": "MoonK SpA",                  # SII: MOONK CORREDORES DE BOLSA DE PRODUCTOS SA
    "91337000-7": "Polpaico S.A.",              # SII: CEMENTO POLPAICO S A
    "76298660-4": "Asfalcura S.A.",             # SII: CONSTRUCTORA ASFALCURA SPA
    "96856860-4": "Elecmetal S.A.",             # SII: ME/ELECMETAL S A
    "96588850-0": "Worley Chile SpA",
    "77569799-7": "Datamart Chile SpA",         # SII: ASESORIAS DATAMARTCHILE SPA
    "77257741-9": "Smart Fit Chile SpA",
}


rut_clientes = {
    "65092388-K": "Coordinador Eléctrico Nacional",
    "76833300-9": "Essbio S.A.",
    "96802690-9": "Masisa S.A.",
    "78127000-8": "Teck Resources Chile Ltda.",
    "97006000-6": "Banco de Crédito e Inversiones",
    "90749000-9": "Falabella S.A.",
    "77078244-9": "Xepelin SpA",          # razón social SII: X CAPITAL SPA
    "76257379-2": "ISA Interchile S.A.",   # razón social SII: INTERCHILE S.A.
    "61219000-3": "Metro S.A.",
    "90266000-3": "Enaex S.A.",
}

df_clientes_rut = df_segmentado[df_segmentado["RUT_DV"].isin(rut_clientes)].copy()

df_clientes_total = df_funcionando_workers[df_funcionando_workers["RUT_DV"].isin(empresas_target)].copy() #saco la lista de los clientes en el DF sin filtrar completos. 

df_clientes_total[df_clientes_total["Segmento"].isna()] #clientes que no fueron segmentados, revisar por qué. 

,RUT_DV,Razón social,Tramo según ventas,Número de trabajadores,Tipo de contribuyente,Actividad económica,Región,Segmento
31768,70017820-K,CAMARA DE COMERCIO DE SANTIAGO A G,12,234,ORG. SIN FINES DE LUCRO,ACTIVIDADES DE ASOCIACIONES EMPRESARIALES Y DE...,XIII REGION METROPOLITANA,None
33797,71500500-K,UNIVERSIDAD MAYOR,13,2095,ORG. SIN FINES DE LUCRO,ENSEÑANZA SUPERIOR EN UNIVERSIDADES PRIVADAS,XIII REGION METROPOLITANA,None
33871,71540800-7,UNIVERSIDAD DE LAS AMERICAS,13,1761,ORG. SIN FINES DE LUCRO,ENSEÑANZA SUPERIOR EN UNIVERSIDADES PRIVADAS,XIII REGION METROPOLITANA,None
94929,76167834-5,SK GODELIUS S.A.,9,36,PERSONA JURIDICA COMERCIAL,ACTIVIDADES DE CONSULTORIA DE INFORMATICA Y DE...,XIII REGION METROPOLITANA,None
199541,76466068-4,SOCIEDAD CONCESIONARIA NUEVO PUDAHUEL S.A.,13,243,PERSONA JURIDICA COMERCIAL,ALQUILER DE BIENES INMUEBLES AMOBLADOS O CON E...,XIII REGION METROPOLITANA,None
452772,77126383-6,TOKU SPA,10,163,PERSONA JURIDICA COMERCIAL,OTRAS ACTIVIDADES DE TECNOLOGIA DE LA INFORMAC...,XIII REGION METROPOLITANA,None
971672,81698900-0,PONTIFICIA UNIVERSIDAD CATOLICA DE CHILE,13,9200,ORG. SIN FINES DE LUCRO,ENSEÑANZA SUPERIOR EN UNIVERSIDADES PRIVADAS,XIII REGION METROPOLITANA,None
972822,83628100-4,SONDA S A,13,2248,PERSONA JURIDICA COMERCIAL,ACTIVIDADES DE CONSULTORIA DE INFORMATICA Y DE...,XIII REGION METROPOLITANA,None
979409,96521680-4,REDBANC S.A.,13,289,PERSONA JURIDICA COMERCIAL,ACTIVIDADES DE CONSULTORIA DE INFORMATICA Y DE...,XIII REGION METROPOLITANA,None
980166,96556930-8,PLUXEE CHILE S.A.,13,205,PERSONA JURIDICA COMERCIAL,OTRAS ACTIVIDADES DE SERVICIOS DE APOYO A LAS ...,XIII REGION METROPOLITANA,None


In [ ]:
#clientes de plutto

df_clientes_total


,RUT_DV,Razón social,Tramo según ventas,Número de trabajadores,Tipo de contribuyente,Actividad económica,Región,Segmento
12564,61219000-3,EMPRESA DE TRANSPORTE DE PASAJEROS METRO S A,13,4987,PERSONA JURIDICA COMERCIAL,TRANSPORTE URBANO Y SUBURBANO DE PASAJEROS VIA...,XIII REGION METROPOLITANA,Utilities / Infraestructura
31768,70017820-K,CAMARA DE COMERCIO DE SANTIAGO A G,12,234,ORG. SIN FINES DE LUCRO,ACTIVIDADES DE ASOCIACIONES EMPRESARIALES Y DE...,XIII REGION METROPOLITANA,None
33797,71500500-K,UNIVERSIDAD MAYOR,13,2095,ORG. SIN FINES DE LUCRO,ENSEÑANZA SUPERIOR EN UNIVERSIDADES PRIVADAS,XIII REGION METROPOLITANA,None
33871,71540800-7,UNIVERSIDAD DE LAS AMERICAS,13,1761,ORG. SIN FINES DE LUCRO,ENSEÑANZA SUPERIOR EN UNIVERSIDADES PRIVADAS,XIII REGION METROPOLITANA,None
37165,76000739-0,ESVAL S.A.,13,853,PERSONA JURIDICA COMERCIAL,"CAPTACION, TRATAMIENTO Y DISTRIBUCION DE AGUA",V REGION VALPARAISO,Utilities / Infraestructura
71697,76101812-4,SOPROLE S.A.,13,1462,PERSONA JURIDICA COMERCIAL,ELABORACION DE PRODUCTOS LACTEOS,XIII REGION METROPOLITANA,Manufactura / Retail
94929,76167834-5,SK GODELIUS S.A.,9,36,PERSONA JURIDICA COMERCIAL,ACTIVIDADES DE CONSULTORIA DE INFORMATICA Y DE...,XIII REGION METROPOLITANA,None
127214,76257379-2,INTERCHILE S.A.,13,122,PERSONA JURIDICA COMERCIAL,TRANSMISION DE ENERGIA ELECTRICA,XIII REGION METROPOLITANA,Mining / Energía
140896,76298660-4,CONSTRUCTORA ASFALCURA SPA,12,433,PERSONA JURIDICA COMERCIAL,CONSTRUCCION DE CARRETERAS Y LINEAS DE FERROCA...,XIII REGION METROPOLITANA,Manufactura / Retail
148239,76320186-4,TECNO FAST S.A.,13,1361,PERSONA JURIDICA COMERCIAL,TERMINACION Y ACABADO DE EDIFICIOS,XIII REGION METROPOLITANA,Mining / Energía


In [50]:
# import json, io, sys

# output_path = "/Users/juanpablozepeda/Proyecto Plutto /Sii folder/scoring_clientes.txt"
# resultados = []

# with open(output_path, "w", encoding="utf-8") as f:
#     for _, row in df_clientes_rut.iterrows():

#         # Capturar el print interno de score_lead
#         buffer = io.StringIO()
#         sys.stdout = buffer

#         resultado = score_lead(
#             company_name = str(row["Razón social"]),
#             rut          = str(row["RUT_DV"]),
#             giro         = str(row["Actividad económica"]),
#             tramo        = str(row["Tramo según ventas"]),
#             region       = str(row["Región"]),
#         )

#         sys.stdout = sys.__stdout__
#         printed = buffer.getvalue()

#         f.write(printed)
#         f.write("JSON completo:\n")
#         f.write(json.dumps(resultado, ensure_ascii=False, indent=2))
#         f.write("\n\n" + "="*55 + "\n\n")

#         resultados.append({
#             "empresa": row["Razón social"],
#             "rut":     row["RUT_DV"],
#             **resultado
#         })
#         print(printed)

# print(f"Resultados guardados en: {output_path}")

In [57]:
import sys, importlib
sys.path.insert(0, "/Users/juanpablozepeda/Proyecto Plutto /Scoring")
import Scoring_ICP
importlib.reload(Scoring_ICP)
from Scoring_ICP import score_lead, score_lead_adj1
import pandas as pd

output_csv = "/Users/juanpablozepeda/Proyecto Plutto /Sii folder/comparacion_scoring.csv"
filas = []

for _, row in df_clientes_total.iterrows():
    empresa   = str(row["Razón social"])
    rut       = str(row["RUT_DV"])
    giro      = str(row["Actividad económica"])
    tramo     = str(row["Tramo según ventas"])
    region    = str(row["Región"])

    for modo, fn in [("normal", score_lead), ("ajustado", score_lead_adj1)]:
        try:
            r = fn(company_name=empresa, rut=rut, giro=giro, tramo=tramo, region=region)
            filas.append({
                "Empresa":    empresa,
                "RUT":        rut,
                "Modo":       modo,
                "Score":      r.get("score"),
                "Vertical":   r.get("vertical"),
                "Dolor":      r.get("pain_point"),
                "Reasoning":  r.get("reasoning"),
            })
            print(f"✓ {empresa} [{modo}] → {r.get('score')}")
        except Exception as e:
            print(f"✗ {empresa} [{modo}] → ERROR: {e}")

df_scoring = pd.DataFrame(filas)
df_scoring.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"\nCSV guardado: {output_csv}")
df_scoring

✓ EMPRESA DE TRANSPORTE DE PASAJEROS METRO S A [normal] → 42
✓ EMPRESA DE TRANSPORTE DE PASAJEROS METRO S A [ajustado] → 83
✓ CAMARA DE COMERCIO DE SANTIAGO A G [normal] → 12
✓ CAMARA DE COMERCIO DE SANTIAGO A G [ajustado] → 39
✓ UNIVERSIDAD MAYOR [normal] → 32
✓ UNIVERSIDAD MAYOR [ajustado] → 39
✓ UNIVERSIDAD DE LAS AMERICAS [normal] → 12
✓ UNIVERSIDAD DE LAS AMERICAS [ajustado] → 39
✓ ESVAL S.A. [normal] → 62
✓ ESVAL S.A. [ajustado] → 72
✓ SOPROLE S.A. [normal] → 42
✓ SOPROLE S.A. [ajustado] → 83
✓ SK GODELIUS S.A. [normal] → 17
✓ SK GODELIUS S.A. [ajustado] → 30
✓ INTERCHILE S.A. [normal] → 65
✓ INTERCHILE S.A. [ajustado] → 72
✓ CONSTRUCTORA ASFALCURA SPA [normal] → 42
✓ CONSTRUCTORA ASFALCURA SPA [ajustado] → 56
✓ TECNO FAST S.A. [normal] → 17
✓ TECNO FAST S.A. [ajustado] → 28
✓ NS AGRO S.A. [normal] → 65
✓ NS AGRO S.A. [ajustado] → 76
✓ BUDA.COM SPA [normal] → 65
✓ BUDA.COM SPA [ajustado] → 67
✓ ALISERVICE S.A. [normal] → 42
✓ ALISERVICE S.A. [ajustado] → 72
✓ SOCIEDAD CONCESIONAR

,Empresa,RUT,Modo,Score,Vertical,Dolor,Reasoning
0,EMPRESA DE TRANSPORTE DE PASAJEROS METRO S A,61219000-3,normal,42,manufactura,Gestión de compliance en cadena de proveedores...,Empresa gran tamaño (Tramo 13 = ventas altas) ...
1,EMPRESA DE TRANSPORTE DE PASAJEROS METRO S A,61219000-3,ajustado,83,OTRO,Evaluación de proveedores y contratistas en tr...,Metro S.A. es regulada por SEC (superintendenc...
2,CAMARA DE COMERCIO DE SANTIAGO A G,70017820-K,normal,12,manufactura,Verificación de miembros y proveedores en red ...,Cámara de Comercio tiene bajo potencial KYB. N...
3,CAMARA DE COMERCIO DE SANTIAGO A G,70017820-K,ajustado,39,otro,Validación de miembros y proveedores para aseg...,Cámara de Comercio es asociación empresarial s...
4,UNIVERSIDAD MAYOR,71500500-K,normal,32,manufactura,Verificación de proveedores y contratistas en ...,Universidad privada grande (tramo 13) sin regu...
...,...,...,...,...,...,...,...
91,EMPRESA DE SERVICIOS EXTERNOS ASOCIACION CHILE...,99579260-5,ajustado,72,OTRO,Evaluación de proveedores y colaboradores en s...,Empresa clasificada como OTRO (servicios de sa...
92,LATAM TRADE CAPITAL S.A.,99595990-9,normal,55,financiero,Verificación de contrapartes y cumplimiento no...,Empresa financiera (servicios financieros no b...
93,LATAM TRADE CAPITAL S.A.,99595990-9,ajustado,63,financiero,Verificación de contrapartes y cumplimiento no...,LATAM TRADE CAPITAL es empresa de servicios fi...
94,SIGDO KOPPERS S A,99598300-1,normal,65,financiero,Verificación de contrapartes y beneficiarios f...,Empresa financiera regulada por CMF con obliga...
